In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
from scipy.interpolate import griddata, interp1d
from mpl_toolkits.mplot3d import Axes3D
import csv
import pandas as pd
from scipy.interpolate import interp1d, UnivariateSpline, CubicSpline
from scipy.signal import savgol_filter

In [ ]:
# 设置matplotlib参数
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['mathtext.rm'] = 'Times New Roman'

In [ ]:
def plot_heatmap_from_data(data_file, output_dir=None):
    """
    从数据文件读取T_forward结果并绘制热图
    
    参数:
    data_file: 数据文件路径
    output_dir: 输出目录，如果为None则使用当前目录
    """
    
    # 读取数据
    try:
        # 尝试使用pandas读取，处理可能的编码问题
        df = pd.read_csv(data_file, sep='\t', encoding='utf-8')
    except UnicodeDecodeError:
        try:
            df = pd.read_csv(data_file, sep='\t', encoding='gbk')
        except:
            df = pd.read_csv(data_file, sep='\t', encoding='latin-1')
    
    # 清理列名（移除可能的特殊字符）
    df.columns = df.columns.str.replace('��', '', regex=False)
    df.columns = df.columns.str.strip()
    
    # 重命名列（如果需要）
    expected_columns = ['r_11(μm)', 'r_12(μm)', 'T_forward']
    if len(df.columns) == 3:
        df.columns = expected_columns
    
    print("数据列名:", df.columns.tolist())
    print("数据形状:", df.shape)
    print("前5行数据:")
    print(df.head())
    
    # 获取唯一的r_11和r_12值
    r_11_unique = sorted(df['r_11(μm)'].unique())
    r_12_unique = sorted(df['r_12(μm)'].unique())
    
    print(f"r_11范围: {r_11_unique}")
    print(f"r_12范围: {r_12_unique}")
    
    # 创建T_forward矩阵
    T_forward_matrix = np.zeros((len(r_11_unique), len(r_12_unique)))
    
    # 填充矩阵
    for _, row in df.iterrows():
        i = r_11_unique.index(row['r_11(μm)'])
        j = r_12_unique.index(row['r_12(μm)'])
        T_forward_matrix[i, j] = row['T_forward']
    
    # 设置matplotlib参数
    plt.rcParams['font.family'] = 'Times New Roman'
    plt.rcParams['xtick.labelsize'] = 12
    plt.rcParams['ytick.labelsize'] = 12
    
    # 绘制热图
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # 使用imshow绘制热图
    im = ax.imshow(T_forward_matrix, cmap='viridis', aspect='auto', origin='lower',
                   vmin=np.min(T_forward_matrix), vmax=np.max(T_forward_matrix))
    
    # 设置坐标轴刻度和标签
    ax.set_xticks(range(len(r_12_unique)))
    ax.set_xticklabels([f"{r:.1f}" for r in r_12_unique])
    ax.set_yticks(range(len(r_11_unique)))
    ax.set_yticklabels([f"{r:.1f}" for r in r_11_unique])
    
    # 设置轴标签和标题
    ax.set_xlabel('r_12 (μm)', fontsize=14)
    ax.set_ylabel('r_11 (μm)', fontsize=14)
    ax.set_title('r_11&r_12 vs Fundamental TE T_forward', fontsize=16, fontweight='bold')
    
    # 添加颜色条
    cbar = plt.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label('T_forward', fontsize=14)
    cbar.ax.tick_params(labelsize=12)
    
    # 在每个格子中显示数值
    for i in range(len(r_11_unique)):
        for j in range(len(r_12_unique)):
            value = T_forward_matrix[i, j]
            # # 根据数值大小选择文字颜色
            # text_color = 'white' if value < (np.max(T_forward_matrix) + np.min(T_forward_matrix))/2 else 'black'
            text_color = 'white'
            text = ax.text(j, i, f'{value:.3f}',
                          ha="center", va="center", color=text_color, 
                          fontsize=8, fontweight='bold')
    
    # 添加网格线
    ax.set_xticks(np.arange(len(r_12_unique))-0.5, minor=True)
    ax.set_yticks(np.arange(len(r_11_unique))-0.5, minor=True)
    ax.grid(which="minor", color="white", linestyle='-', linewidth=1)
    
    plt.tight_layout()
    
    # 保存图片
    if output_dir is None:
        output_dir = os.path.dirname(data_file) if os.path.dirname(data_file) else '.'
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # 生成输出文件名
    base_name = os.path.splitext(os.path.basename(data_file))[0]
    heatmap_path = os.path.join(output_dir, f"{base_name}_heatmap.png")
    
    plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
    print(f"热图已保存到: {heatmap_path}")
    
    # 显示图片
    plt.show()
    
    # 打印统计信息
    print(f"\n统计信息:")
    print(f"T_forward最大值: {np.max(T_forward_matrix):.6f}")
    print(f"T_forward最小值: {np.min(T_forward_matrix):.6f}")
    print(f"T_forward平均值: {np.mean(T_forward_matrix):.6f}")
    
    # 找到最优参数
    max_idx = np.unravel_index(np.argmax(T_forward_matrix), T_forward_matrix.shape)
    optimal_r11 = r_11_unique[max_idx[0]]
    optimal_r12 = r_12_unique[max_idx[1]]
    max_value = T_forward_matrix[max_idx]
    
    print(f"最优参数: r_11 = {optimal_r11:.1f} μm, r_12 = {optimal_r12:.1f} μm")
    print(f"最大T_forward值: {max_value:.6f}")
    
    return T_forward_matrix, r_11_unique, r_12_unique

In [ ]:
def create_contour_plot(data_file, output_dir=None):
    """
    创建等高线图作为补充
    """
    # 重复数据读取过程（简化版）
    try:
        df = pd.read_csv(data_file, sep='\t', encoding='utf-8')
    except UnicodeDecodeError:
        try:
            df = pd.read_csv(data_file, sep='\t', encoding='gbk')
        except:
            df = pd.read_csv(data_file, sep='\t', encoding='latin-1')
    
    df.columns = df.columns.str.replace('��', '', regex=False).str.strip()
    if len(df.columns) == 3:
        df.columns = ['r_11(μm)', 'r_12(μm)', 'T_forward']
    
    r_11_unique = sorted(df['r_11(μm)'].unique())
    r_12_unique = sorted(df['r_12(μm)'].unique())
    
    T_forward_matrix = np.zeros((len(r_11_unique), len(r_12_unique)))
    for _, row in df.iterrows():
        i = r_11_unique.index(row['r_11(μm)'])
        j = r_12_unique.index(row['r_12(μm)'])
        T_forward_matrix[i, j] = row['T_forward']
    
    # 创建网格
    R11, R12 = np.meshgrid(r_12_unique, r_11_unique)
    
    # 绘制等高线图
    plt.figure(figsize=(10, 8))
    contour = plt.contour(R11, R12, T_forward_matrix, levels=15, colors='black', linewidths=0.5)
    contourf = plt.contourf(R11, R12, T_forward_matrix, levels=20, cmap='viridis', alpha=0.8)
    
    plt.clabel(contour, inline=True, fontsize=8, fmt='%.3f')
    plt.colorbar(contourf, label='T_forward')
    
    plt.xlabel('r_12 (μm)', fontsize=14)
    plt.ylabel('r_11 (μm)', fontsize=14)
    plt.title('T_forward Contour Plot', fontsize=16)
    plt.grid(True, alpha=0.3)
    
    if output_dir is None:
        output_dir = os.path.dirname(data_file) if os.path.dirname(data_file) else '.'
    
    base_name = os.path.splitext(os.path.basename(data_file))[0]
    contour_path = os.path.join(output_dir, f"{base_name}_contour.png")
    plt.savefig(contour_path, dpi=300, bbox_inches='tight')
    print(f"等高线图已保存到: {contour_path}")
    plt.show()

In [ ]:
# 详细数据显示
data_file = "D:/simulation/Simulation Project/results/r_11_r_12_scan/T_forward_sweep_results.txt"
output_directory = None
T_matrix, r11_vals, r12_vals = plot_heatmap_from_data(data_file, output_directory)
# print("\n" + "="*50)
# print("开始绘制等高线图...")
# create_contour_plot(data_file, output_directory)
# print("\n所有图表已生成完成！")

In [ ]:
def plot_3d_surface_from_data(data_file, output_dir=None):
    """
    从数据文件读取T_forward结果并绘制三维表面图
    
    参数:
    data_file: 数据文件路径
    output_dir: 输出目录，如果为None则使用当前目录
    """
    
    # 读取数据
    try:
        df = pd.read_csv(data_file, sep='\t', encoding='utf-8')
    except UnicodeDecodeError:
        try:
            df = pd.read_csv(data_file, sep='\t', encoding='gbk')
        except:
            df = pd.read_csv(data_file, sep='\t', encoding='latin-1')
    
    # 清理列名
    df.columns = df.columns.str.replace('��', '', regex=False).str.strip()
    if len(df.columns) == 3:
        df.columns = ['r_11', 'r_12', 'T_forward']
    
    print("数据列名:", df.columns.tolist())
    print("数据形状:", df.shape)
    print("数据范围:")
    print(f"r_11: {df['r_11'].min():.1f} - {df['r_11'].max():.1f}")
    print(f"r_12: {df['r_12'].min():.1f} - {df['r_12'].max():.1f}")
    print(f"T_forward: {df['T_forward'].min():.6f} - {df['T_forward'].max():.6f}")
    
    # 提取数据
    r_11 = df['r_11'].values
    r_12 = df['r_12'].values
    T_forward = df['T_forward'].values
    
    # 生成更细致的网格用于插值
    r_11_lin = np.linspace(r_11.min(), r_11.max(), 100)
    r_12_lin = np.linspace(r_12.min(), r_12.max(), 100)
    R_12_grid, R_11_grid = np.meshgrid(r_12_lin, r_11_lin)
    
    # 使用三次插值生成平滑表面
    T_grid = griddata((r_12, r_11), T_forward, (R_12_grid, R_11_grid), method='cubic')
    
    # 处理插值后的NaN值
    mask = ~np.isnan(T_grid)
    if not np.all(mask):
        # 使用线性插值填补NaN
        T_grid_linear = griddata((r_12, r_11), T_forward, (R_12_grid, R_11_grid), method='linear')
        T_grid = np.where(mask, T_grid, T_grid_linear)
    
    # 设置matplotlib参数
    plt.rcParams['font.family'] = 'Times New Roman'
    plt.rcParams['xtick.labelsize'] = 12
    plt.rcParams['ytick.labelsize'] = 12
    
    # 创建3D图形
    fig = plt.figure(figsize=(14, 10))
    
    # # 3D表面图
    # ax1 = fig.add_subplot(221, projection='3d')
    # surf1 = ax1.plot_surface(R_12_grid, R_11_grid, T_grid, 
    #                         cmap='viridis', alpha=0.9, edgecolor='none')
    # ax1.set_xlabel('r_12 (μm)', fontsize=12)
    # ax1.set_ylabel('r_11 (μm)', fontsize=12)
    # ax1.set_zlabel('T_forward', fontsize=12)
    # ax1.set_title('3D Surface: T_forward vs r_11 & r_12', fontsize=14)
    # cbar1 = fig.colorbar(surf1, ax=ax1, shrink=0.5, aspect=10)
    # cbar1.set_label('T_forward', fontsize=10)
    
    # # 3D线框图
    # ax2 = fig.add_subplot(222, projection='3d')
    # surf2 = ax2.plot_wireframe(R_12_grid, R_11_grid, T_grid, 
    #                           color='blue', alpha=0.7, linewidth=0.8)
    # ax2.set_xlabel('r_12 (μm)', fontsize=12)
    # ax2.set_ylabel('r_11 (μm)', fontsize=12)
    # ax2.set_zlabel('T_forward', fontsize=12)
    # ax2.set_title('3D Wireframe: T_forward vs r_11 & r_12', fontsize=14)
    
    # # 3D散点图（原始数据点）
    # ax3 = fig.add_subplot(223, projection='3d')
    # scatter = ax3.scatter(r_12, r_11, T_forward, c=T_forward, 
    #                      cmap='viridis', s=50, alpha=0.8)
    # ax3.set_xlabel('r_12 (μm)', fontsize=12)
    # ax3.set_ylabel('r_11 (μm)', fontsize=12)
    # ax3.set_zlabel('T_forward', fontsize=12)
    # ax3.set_title('3D Scatter: Original Data Points', fontsize=14)
    # cbar3 = fig.colorbar(scatter, ax=ax3, shrink=0.5, aspect=10)
    # cbar3.set_label('T_forward', fontsize=10)
    
    # 等高线图（2D投影）
    ax4 = fig.add_subplot(224)
    contour = ax4.contour(R_12_grid, R_11_grid, T_grid, levels=15, colors='black', linewidths=0.5)
    contourf = ax4.contourf(R_12_grid, R_11_grid, T_grid, levels=20, cmap='viridis', alpha=0.8)
    ax4.clabel(contour, inline=True, fontsize=8, fmt='%.3f')
    ax4.set_xlabel('r_12 (μm)', fontsize=12)
    ax4.set_ylabel('r_11 (μm)', fontsize=12)
    ax4.set_title('2D Contour Projection', fontsize=14)
    cbar4 = fig.colorbar(contourf, ax=ax4)
    cbar4.set_label('T_forward', fontsize=10)

    plt.tight_layout()
    
    # 保存图片
    if output_dir is None:
        output_dir = os.path.dirname(data_file) if os.path.dirname(data_file) else '.'
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    base_name = os.path.splitext(os.path.basename(data_file))[0]
    plot_path = os.path.join(output_dir, f"{base_name}_3D_analysis.png")
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    print(f"3D分析图已保存到: {plot_path}")
    plt.show()
    
    return fig

In [ ]:
def loss_plot_3d_surface_from_data(data_file, output_dir=None):
    """
    从数据文件读取T_forward结果并绘制三维表面图
    
    参数:
    data_file: 数据文件路径
    output_dir: 输出目录，如果为None则使用当前目录
    """
    
    # 读取数据
    try:
        df = pd.read_csv(data_file, sep='\t', encoding='utf-8')
    except UnicodeDecodeError:
        try:
            df = pd.read_csv(data_file, sep='\t', encoding='gbk')
        except:
            df = pd.read_csv(data_file, sep='\t', encoding='latin-1')
    
    # 清理列名
    df.columns = df.columns.str.replace('��', '', regex=False).str.strip()
    if len(df.columns) == 3:
        df.columns = ['r_11', 'r_12', 'T_forward']
    
    # 提取数据
    r_11 = df['r_11'].values
    r_12 = df['r_12'].values
    T_forward = df['T_forward'].values
    
    # 生成更细致的网格用于插值
    r_11_lin = np.linspace(r_11.min(), r_11.max(), 100)
    r_12_lin = np.linspace(r_12.min(), r_12.max(), 100)
    R_12_grid, R_11_grid = np.meshgrid(r_12_lin, r_11_lin)
    
    # 使用三次插值生成平滑表面
    T_grid = griddata((r_12, r_11), T_forward, (R_12_grid, R_11_grid), method='cubic')
    
    # 处理插值后的NaN值
    mask = ~np.isnan(T_grid)
    if not np.all(mask):
        # 使用线性插值填补NaN
        T_grid_linear = griddata((r_12, r_11), T_forward, (R_12_grid, R_11_grid), method='linear')
        T_grid = np.where(mask, T_grid, T_grid_linear)

    loss = -10 * np.log10(T_grid)
    # 设置matplotlib参数
    plt.rcParams['font.family'] = 'Times New Roman'
    plt.rcParams['xtick.labelsize'] = 12
    plt.rcParams['ytick.labelsize'] = 12
    
    # 创建3D图形
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(224)
    levels = np.linspace(0.8, loss.max(), 30)
    contour = ax.contour(R_12_grid, R_11_grid, loss, levels=levels, colors='black', linewidths=0.5)
    contourf = ax.contourf(R_12_grid, R_11_grid, loss, levels=levels, cmap='viridis_r', alpha=0.8)
    ax.clabel(contour, inline=True, fontsize=8, fmt='%.3f')
    ax.set_xlabel(r'$\mathbf{r}_{\boldsymbol{12}}$ (μm)', fontsize=12,fontweight='bold')
    ax.set_ylabel(r'$\mathbf{r}_{\boldsymbol{11}}$ (μm)', fontsize=12,fontweight='bold')
    # ax.set_title('Loss', fontsize=16,fontweight='bold')
    # cbar = fig.colorbar(contourf, ax=ax)
    # cbar.set_label('loss(dB)', fontsize=16,fontweight='bold')
    # plt.tight_layout()
    
    # 保存图片
    if output_dir is None:
        output_dir = os.path.dirname(data_file) if os.path.dirname(data_file) else '.'
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # base_name = os.path.splitext(os.path.basename(data_file))[0]
    # plot_path = os.path.join(output_dir, f"{base_name}_3D_analysis.png")
    # plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    # print(f"3D分析图已保存到: {plot_path}")
    plt.show()
    
    return fig

In [ ]:
data_file = "D:/simulation/Simulation Project/results/r_11_r_12_scan/T_forward_sweep_results.txt"
output_directory = None
print("1. 生成综合3D分析图...")
fig1 = plot_3d_surface_from_data(data_file, output_directory)
# print("\n2. 生成单独的大尺寸3D表面图...")
# fig2 = plot_single_3d_surface(data_file, output_directory, view_angle=(20, 45))
# print("\n所有3D图表已生成完成！")

In [ ]:
data_file = "D:/simulation/Simulation Project/simulation/LD-PWB-SMF/results/r_11_r_12_scan/T_forward_sweep_results.txt"
fig2 = loss_plot_3d_surface_from_data(data_file, output_directory)

In [ ]:
data = pd.read_csv("d:/simulation/Simulation Project/results/l1_scan/T_forward_sweep_results.txt", sep='\t')
target_x = 16
x = data['l1'].values
y = -10 * np.log10(data['T_forward'].values)
plt.plot(x, y, '-', color='blue', linewidth=2)
plt.scatter(x, y, color='red', s=10, zorder=5, linewidth=2)
plt.axvline(x=target_x, color='red', linestyle='--', alpha=0.5, linewidth=1)
plt.title('Loss', fontsize=16,fontweight='bold')
plt.xlabel(r'$\mathbf{l}_{\boldsymbol{1}}$ (μm)', fontsize=16,fontweight='bold')
plt.ylabel('loss (dB)', fontsize=16,fontweight='bold')
for spine in plt.gca().spines.values():
    spine.set_linewidth(2)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../results/Pictures/l1-loss.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
data = pd.read_csv("d:/simulation/Simulation Project/results/r_2_scan/T_forward_sweep_results.txt", sep='\t')
x = data['r_2'].values
y = -10 * np.log10(data['T_forward'].values)
cs = CubicSpline(x, y)
x_interp = np.linspace(x.min(), x.max(), 200)
y_cs = cs(x_interp)
plt.plot(x_interp, y_cs, '-', color='blue', linewidth=2)
# plt.scatter(x, y, color='red', s=10, zorder=5, linewidth=2)
plt.title('Loss', fontsize=16,fontweight='bold')
plt.xlabel(r'$\mathbf{r}_{\boldsymbol{2}}$ (μm)', fontsize=16,fontweight='bold')
plt.ylabel('loss (dB)', fontsize=16,fontweight='bold')
for spine in plt.gca().spines.values():
    spine.set_linewidth(2)
plt.grid(True, alpha=0.3)
plt.text(x=1.5, y=3.6, s=r'$\mathbf{r}_{\boldsymbol{11}}$=0.6 μm', fontsize=16, color='black')
plt.text(x=1.5, y=3.3, s=r'$\mathbf{r}_{\boldsymbol{12}}$=1.5 μm', fontsize=16, color='black')
plt.text(x=1.5, y=3.0, s=r'$\mathbf{l}_{\boldsymbol{1}}$=16 μm', fontsize=16, color='black')
plt.tight_layout()
plt.savefig("../results/Pictures/r_2-loss.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 拟合曲线
data = pd.read_csv("d:/simulation/Simulation Project/results/l1_scan/T_forward_sweep_results.txt", sep='\t')
x = data['l1'].values
y = data['T_forward'].values
cs = CubicSpline(x, y)
x_interp = np.linspace(x.min(), x.max(), 200)
y_cs = cs(x_interp)
target_l1 = 16
exact_match = x == target_l1
if np.any(exact_match):
    target_x = x[exact_match][0]
    target_y = y[exact_match][0]
    print(f"找到精确匹配: l1={target_x}, T_forward={target_y}")
else:
    print("没有找到精确匹配的点，使用最接近的点")
    # 找到最接近的点
    closest_idx = np.argmin(np.abs(x - target_l1))
    target_x = x[closest_idx]
    target_y = y[closest_idx]
    print(f"最接近的点: l1={target_x}, T_forward={target_y}")
plt.scatter(target_x, target_y, color='red', s=30, zorder=5, linewidth=2, label=f'l1={target_x}μm')
plt.axvline(x=target_x, color='red', linestyle='--', alpha=0.5, linewidth=1)
plt.plot(x_interp, y_cs, '-', color='blue', linewidth=2, label='curve')
plt.title('l1-T_forward')
plt.xlabel('l1')
plt.ylabel('T_forward')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("./l1-T_forward.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# r
data = pd.read_csv("d:/simulation/Simulation Project/results/r_scan_r2=1.1/T_forward_sweep_results.txt", sep='\t')
x = data['r'].values
y = -10 * np.log10(data['T_forward'].values)
cs = CubicSpline(x, y)
x_interp = np.linspace(x.min(), x.max(), 200)
y_cs = cs(x_interp)
plt.plot(x_interp, y_cs, '-', color='blue', linewidth=2)
# plt.scatter(x, y, color='red', s=10, zorder=5, linewidth=2)
plt.title('Loss', fontsize=16,fontweight='bold')
plt.xlabel('r (μm)', fontsize=16,fontweight='bold')
plt.ylabel('loss (dB)', fontsize=16,fontweight='bold')
plt.grid(True, alpha=0.3)
for spine in plt.gca().spines.values():
    spine.set_linewidth(2)
plt.text(x=102, y=11, s=r'$\mathbf{r}_{\boldsymbol{2}}$=1.1 μm', fontsize=16, color='black')
plt.tight_layout()
plt.savefig("../results/Pictures/r-loss.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# l2
data = pd.read_csv("d:/simulation/Simulation Project/results/l2_scan/T_forward_sweep_results.txt", sep='\t')
x = data['l2'].values
y = -10 * np.log10(data['T_forward'].values) + 0.2
cs = CubicSpline(x, y)
x_interp = np.linspace(x.min(), x.max(), 200)
y_cs = cs(x_interp)
plt.plot(x_interp, y_cs, '-', color='blue', linewidth=2)
# plt.scatter(x, y, color='red', s=10, zorder=5, linewidth=2)
plt.title('Loss', fontsize=16,fontweight='bold')
plt.xlabel(r'$\mathbf{l}_{\boldsymbol{2}}$ (μm)', fontsize=16,fontweight='bold')
plt.ylabel('loss (dB)', fontsize=16,fontweight='bold')
for spine in plt.gca().spines.values():
    spine.set_linewidth(2)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../results/Pictures/l2-loss.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# r3
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize

data = pd.read_csv("d:/simulation/Simulation Project/simulation/LD-PWB-SMF/results/r_3_scan/T_forward_sweep_results.txt", sep='\t')
x = data['r_3'].values
# 原始点的 loss 值
y_points = -10 * np.log10(data['T_forward'].values) + 0.27
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['xtick.labelsize'] = 32
plt.rcParams['ytick.labelsize'] = 32
# 平滑曲线（插值）
cs = CubicSpline(x, y_points)
x_interp = np.linspace(x.min(), x.max(), 1000)
y_cs = cs(x_interp)

fig, ax = plt.subplots(figsize=(12,10))

# 1) 用 LineCollection 按 loss 值给曲线着色（loss 在插值点上）
loss_interp = np.interp(x_interp, x, y_points)
points = np.array([x_interp, y_cs]).T.reshape(-1,1,2)
segments = np.concatenate([points[:-1], points[1:]], axis=1)
lc = LineCollection(segments, array=loss_interp[:-1], cmap='rainbow_r',
                    norm=Normalize(vmin=loss_interp.min(), vmax=loss_interp.max()),
                    linewidth=2, zorder=2)
ax.add_collection(lc)

# 2) 用 hollow 小方框显示原始数据（facecolors='none' 使方框中间透明，曲线可见穿过）
ax.scatter(x, y_points, marker='s', s=80, facecolors='none', edgecolors='black',
           linewidths=0.8, zorder=3)

# 标出指定 loss 值处（例如 0.52 和 0.36）为蓝色点（若没精确点则取最近插值点）
# for val in (0.52, 0.36):
#     idx = np.argmin(np.abs(loss_interp - val))
#     ax.scatter(x_interp[idx], y_cs[idx], color='blue', s=50, zorder=4)
#     ax.annotate(f'{val:.2f}', (x_interp[idx], y_cs[idx]), xytext=(5,5), textcoords='offset points', color='blue', fontsize=9)

ax.set_xlim(x_interp.min() - 0.2, x_interp.max() + 0.2)
ax.set_ylim(0.36, 0.52)

# ax.set_title('Loss', fontsize=16, fontweight='bold')
ax.set_xlabel(r'$\mathbf{r}_{\boldsymbol{3}}$ (μm)', fontsize=30, fontweight='bold')
ax.set_ylabel('loss (dB)', fontsize=30, fontweight='bold')
for spine in ax.spines.values():
    spine.set_linewidth(2)
ax.grid(True, alpha=0.3)
ax.legend(frameon=False)
plt.show()